# 13. The paired tests: two realised tracking errors on the same months

**What this notebook does.** It compares, two at a time, figures that notebook 09 measured on the same 306 months, with a test built for figures that share their months, and writes four tables. It reads only the tables committed in `results/20260928/`, needs no extraction and runs on any machine with a clone of the repository. It was added on 18 November 2026, after every recorded run of the study, and changes no result that notebooks 01 to 11 judged.

**Why it comes here.** Two paths run on the same months move together month by month, so the difference between their figures is far less noisy than either figure, and a test on the paired months measures that difference against its own uncertainty.

**Terms used in this notebook.**

- *market capitalisation*: price times shares outstanding, at the end of the month before.
- *benchmark*: the universe of notebook 02, each stock weighted by its market capitalisation at the start of the month.
- *active return*: the portfolio's return minus the benchmark's in the same month.
- *basis point*: a hundredth of a percentage point.
- *tracking error*: the standard deviation of the monthly active return, times the square root of 12, here in basis points a year. The realised tracking error is the one measured over the months a path has run.
- *covariance matrix*: the table of the stocks' variances and of their covariances with each other.
- *factor*: a common source of return, here one of French's five factors or his momentum factor.
- *covariance estimator*: one way of estimating the covariance matrix from three years of daily returns: the sample covariance, the Ledoit-Wolf estimator, the factor estimator, built on the six factors, and the principal-component estimator.
- *turnover*: the month's one-way trading as a share of the portfolio. Chosen turnover is the part the optimiser chooses, which a cap limits.
- *path*: one covariance estimator under one constraint set at one cap on chosen turnover, run month by month from July 1999 to December 2024. The *mandate* is constraint set C3 of notebook 08.
- *expectation*: a test of a result fixed in advance. Its *verdict* is met or not met.
- *standard error*: the standard deviation an estimate would have across samples of the same length, its uncertainty.
- *paired test*: a test of the difference between two figures measured on the same months, which uses what the two series share month by month. Here it is the variance test of Ledoit and Wolf (2011), copied unchanged from the repository of Optimal versus naive diversification as `src/slt/source/paired.py`.
- *circular block bootstrap*: a way of measuring the uncertainty of a figure from new samples of the same length taken from the months, in blocks of 12 consecutive months, the last month followed by the first, so that the correlation of nearby months is kept. The *seed* is the number that starts the random choice of blocks, so that a rerun gives the same samples.
- *p-value*: from the bootstrap, the probability of a difference at least as far from zero as the one observed if the two variances were equal.
- *95 per cent interval*: the range around the difference of the two tracking errors that the bootstrap gives at the 95 per cent level. A difference whose interval excludes zero is statistically significant at that level.
- *critical value*: the bootstrap's 95th percentile of the difference measured in its own standard errors. The interval is the difference plus and minus the critical value times the standard error.
- *detectable difference*: the difference that the test would find statistically significant at that level about four times in five, the critical value plus 0.84, times the standard error.
- *Holm's correction*: an adjustment of the p-values of a family of comparisons for their number. The smallest p-value is multiplied by the family's size, the next by one less, and each adjusted value is at least the one before it (Holm, 1979).
- *size*: the share of pairs with equal variances that a test rejects at the 5 per cent level, which should be close to 5 per cent.
- *Student-t*: a distribution with fatter tails than the normal one. Its *degrees of freedom* are the parameter that sets how fat, and with 5 the tails are fat.

**What I expect to see, and where it comes from.** The settings were set on 18 November 2026, after every recorded run, and are in `src/slt/constants.py` under `PAIRED_`: blocks of 12 months, 2,000 samples, the level 0.95, the seed 20261009 of Optimal versus naive diversification, and a power of 0.80 for the detectable difference.

- *P1*, the one expectation. In notebook 13 the size check runs before the seven comparisons: on 1,000 simulated pairs of 306 months with equal variances, Student-t shocks with 5 degrees of freedom and a monthly standard deviation of 0.23 per cent, about 80 basis points a year, at a correlation of 0.99 and again at 0.80, the share rejected at 5 per cent is at most 6.5 per cent in each. Below 3.5 per cent at both settings, the text says that the test rejects less often than its level, that an interval that excludes zero stands, and that an interval that includes zero is weaker evidence of no difference than a test of exact size would give.
- *Family A*, with no verdict: the sample, factor and principal-component estimators each against Ledoit-Wolf under the mandate at the 2 per cent cap, with Holm's correction across the three. Its first row bears on the second clause of E4.
- *Family B*, with no verdict: on the Ledoit-Wolf and the factor estimators, the 4 per cent cap and no cap each against the 0.5 per cent cap, with no correction. Its rows bear on the second part of E5.
- *The factor estimator without two months*, added on 19 November 2026, with no verdict: the two rows of family B on the factor estimator again, without July and December 2008, beside the share of the difference between the two variances that those two months carry.
- *The other block lengths*: the seven comparisons again at blocks of 3, 6 and 24 months, with the same seed and 2,000 samples, and Holm's correction across family A at each length. No order of commits is claimed for these lengths. The block is fixed at 12 months, where Ledoit and Wolf (2008, Algorithm 3.1) choose its length from the data. At blocks of 3, 6 and 24 months, added on 19 November 2026, after the study's results were known, with no verdict, the sample estimator's difference from Ledoit-Wolf has intervals of 0.81 to 2.57, 0.84 to 2.55 and 0.61 to 2.77, with Holm p-values of 0.003, 0.0015 and 0.0495, and none of the other six differences is statistically significant at the 5 per cent level at any length, where they could detect differences of 7.1 to 21.1 basis points a year four times in five. The sample estimator's difference is therefore statistically significant at the 5 per cent level at all four lengths, tested on its own. At 24 months its Holm p-value is 0.0495, and 2,000 samples cannot place it on either side of 0.05.

E4 and E5 keep the verdicts of notebook 09, which compare two point estimates as the expectations were fixed before the run. Running time: about two minutes.

In [ ]:
import pathlib
import sys
import time

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd

from slt import constants as C
from slt import extraction as X
from slt import pairs as PR
from slt.source import paired as PT

pd.set_option("display.width", 200)
S = PR.settings()
print(f"set on {C.PAIRED_TEST_DATE}: blocks of {S['block']} months, {S['draws']:,} draws, level {S['level']}, "
      f"seed {C.PAIRED_SEED}, power {C.PAIRED_DETECTABLE_POWER} for the detectable difference")

## 1. The active returns as committed

The table notebook 09 committed, one column for each of the twelve paths under the mandate, read from the repository with its SHA-256, so that this run can be tied to the commit. Every pair tested has a value in every month.

In [ ]:
active, sha = PR.read_active(ROOT)
print(f"{PR.ACTIVE_FILE}: {len(active)} months, {active.index[0]} to {active.index[-1]}, {active.shape[1]} paths, SHA-256 {sha}")
assert len(active) == C.EVAL_MONTHS and list(active.columns) == list(C.MONTHLY_ACTIVE_COLUMNS)
pairs = list(C.PAIRED_FAMILY_A) + list(C.PAIRED_FAMILY_B)
for a, b in pairs:
    assert active["active_return_" + C.PATH_LABELS[a]].notna().all() and active["active_return_" + C.PATH_LABELS[b]].notna().all()
print(f"{len(pairs)} comparisons, every month present in each")

## 2. P1, the size of the test, before the seven comparisons

One thousand pairs of 306 months are simulated with equal variances at each of the two correlations, and the test is run on each pair with the settings above. The share of pairs rejected at 5 per cent by the p-value is the verdict, and the share of intervals that exclude zero is reported beside it.

In [ ]:
t0 = time.time()
size = PR.size_table()
print(size.T.to_string(header=False))
p1 = (size["P1"] == "met").all()
for _, r in size.iterrows():
    note = ", below 3.5 per cent: the test rejects less often than its level" if r["share rejected"] < C.PAIRED_SIZE_CONSERVATIVE_BELOW else ""
    print(f"correlation {r['correlation']}: {r['rejected by p-value']} of {r['pairs']:,} pairs rejected at 5 per cent, "
          f"{100 * r['share rejected']:.1f} per cent{note} | {100 * r['share of intervals excluding zero']:.1f} per cent of the intervals exclude zero")
print(f"P1: {'met' if p1 else 'not met'}, the share at most {100 * C.PAIRED_SIZE_MAX:.1f} per cent in each case | {time.time() - t0:.0f} s")

## 3. Families A and B, from the committed active returns

Each difference is the first path's realised tracking error minus the second's, in basis points a year. The seed is the same for every comparison, so a rerun gives the same table.

In [ ]:
table = PR.paired_table(active)
show = ["comparison", "label", "sd 1", "sd 2", "difference", "standard error", "critical value", "lower", "upper",
        "detectable difference", "correlation", "p-value", "Holm p-value"]
print(table[show].to_string(index=False, float_format=lambda x: f"{x:.3f}"))
for _, r in table.iterrows():
    holm = "" if r["comparison"] == "B" else f", {r['Holm p-value']:.3f} with Holm's correction"
    print(f"{r['label']}: {r['difference']:+.2f} bp a year, interval {r['lower']:+.2f} to {r['upper']:+.2f}, p {r['p-value']:.3f}{holm}, "
          f"detectable {r['detectable difference']:.2f}")

## 4. The factor estimator's two rows without July and December 2008

The critical values of the factor estimator's two rows in family B are far above those of the other five. This section shows where that comes from. For each of the two rows it measures the share of the difference between the two variances that July and December 2008 carry, each path's months measured from its own mean over all 306 months, and runs the test again on the other 304 months with the same settings and seed.

In [ ]:
left_out = PR.left_out_table(active)
share = "share of the difference between the two variances in those months"
for _, r in left_out.iterrows():
    print(f"{r['label']}: {100 * r[share]:.1f} per cent of the difference between the two variances in {r['months left out']}")
    print(f"  without them, {r['months']} months: {r['difference']:+.2f} bp a year, standard error {r['standard error']:.2f}, "
          f"critical value {r['critical value']:.2f}, interval {r['lower']:+.2f} to {r['upper']:+.2f}, p {r['p-value']:.3f}")

## 5. The seven comparisons at blocks of 3, 6 and 24 months, added on 19 November 2026

The block length is fixed at 12 months. Ledoit and Wolf (2008, Algorithm 3.1) choose it from the data instead, by simulation. This section, added on 19 November 2026, after the study's results were known, with no verdict, runs the seven comparisons again at blocks of 3, 6 and 24 months, with the same seed and 2,000 samples, and Holm's correction across family A at each length. With 2,000 samples a p-value moves in steps of one in 2,001, and the error of the bootstrap's own sampling is printed beside each Holm value of family A.

In [ ]:
other = PR.other_blocks_table(active)
print(other[["block"] + show].to_string(index=False, float_format=lambda x: f"{x:.3f}"))
for _, r in other[other["comparison"] == "A"].iterrows():
    n = int(r["draws"])
    se = r["Holm p-value"] / r["p-value"] * (r["p-value"] * (1 - r["p-value"]) / n) ** 0.5
    print(f"blocks of {r['block']}: {r['label']}: interval {r['lower']:+.2f} to {r['upper']:+.2f}, "
          f"Holm p {r['Holm p-value']:.4f}, bootstrap error of that value about {se:.4f}")
excl = ((other["lower"] > 0) | (other["upper"] < 0)).groupby(other["block"]).sum()
print("intervals that exclude zero, by block length:", {int(k): int(v) for k, v in excl.items()})

## 6. What is committed

Four tables go into `results/20260928/`: `paired_tests.csv`, the seven comparisons with every column of the test, `paired_size_check.csv`, P1 at its two correlations, `paired_tests_without_two_months.csv`, the two rows of section 4, and, added on 19 November 2026, `paired_tests_other_blocks.csv`, the seven comparisons at each of the three other block lengths of section 5. None is by month, and no table of an earlier notebook changes.

In [ ]:
for p_ in [X.write_results("paired_tests.csv", frame=table), X.write_results("paired_size_check.csv", frame=size),
           X.write_results("paired_tests_without_two_months.csv", frame=left_out),
           X.write_results("paired_tests_other_blocks.csv", frame=other)]:
    print("written:", p_.relative_to(ROOT))
print(f"active returns tested: SHA-256 {sha}")

## What was learned, and what could be wrong

What the notebook establishes: the size of the test on series of the study's length, tails and correlation, and, for each pair of paths the write-up compares, the difference of their realised tracking errors with an interval measured on their shared months.

What could still be wrong. The interval is the bootstrap's, and on 306 months a block of 12 months is one of 25, so months of large active returns weigh on the critical value. On the factor estimator two months of 2008, July and December, carry 97 and 99 per cent of the difference between the two variances in the two rows of family B, whose critical values are about 5.9 against 2.2 to 2.4 on the other five, and section 4 shows that without those months the critical value is 2.1 and the change about zero. The test compares variances and says nothing about the mean active return. And the seven comparisons were chosen after the results were known, from the sentences of the write-up that compared two tracking errors, so they are a check of those sentences and not a search among all the pairs.